# Task 1.1 — Dataset Exploration & Cleaning

Goal: characterise the dataset, surface any feature that could leak the source class, and design a deterministic cleaning pipeline that downstream tasks can rely on. Outputs feed `solution/clean.py` and report §1.1.

Dataset schema (per PDF):
- labeled splits (`train`, `calibration`, `calibration_augmented`, `validation`, `validation_augmented`): `image: binary, source_class: int8`
- predict split (`predict`): `row_id: int32, image: binary`

Labels: 0=real; 1..5 collapse to 1=ai_generated.

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import sys
import io
from pathlib import Path

ROOT = Path.cwd().parent
SOLUTION = ROOT / "solution"
sys.path.insert(0, str(SOLUTION))

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from PIL import Image

from _lib import seed
from _lib.data import binarize_label

seed.set_deterministic(0)

DATA = SOLUTION / "data"
print("data root:", DATA, "exists:", DATA.exists())

CLASS_NAMES = ["real", "SD2.1", "SDXL", "SD3", "DALL-E3", "Midjourney"]

data root: c:\Users\Agando\Documents\Projects\AI-Image-Detection-AMLS-Project\solution\data exists: True


## 1. Schema sanity

Confirm column names, row counts, and that one image decodes.

In [2]:
splits = ["train", "calibration", "calibration_augmented",
          "validation", "validation_augmented", "predict"]

for s in splits:
    files = sorted((DATA / s).glob("*.parquet"))
    pf = pq.ParquetFile(files[0])
    print(f"{s:25s} files={len(files)}  rows/file0={pf.metadata.num_rows}  cols={pf.schema_arrow.names}")

# decode one row from train/0
pf = pq.ParquetFile(sorted((DATA / "train").glob("*.parquet"))[0])
batch = next(pf.iter_batches(batch_size=1, columns=["image", "source_class"]))
im = Image.open(io.BytesIO(batch["image"][0].as_py()))
print("sample:", im.size, im.mode, im.format, "class=", batch["source_class"][0].as_py())

train                     files=7  rows/file0=4242  cols=['image', 'source_class']
calibration               files=2  rows/file0=962  cols=['image', 'source_class']
calibration_augmented     files=2  rows/file0=962  cols=['image', 'source_class']
validation                files=2  rows/file0=562  cols=['image', 'source_class']
validation_augmented      files=2  rows/file0=562  cols=['image', 'source_class']
predict                   files=1  rows/file0=100  cols=['row_id', 'image']
sample: (320, 320) RGB JPEG class= 2


## 2. Class distribution

Count source classes per split (memory-friendly: only read `source_class`). Then collapse to binary.

In [ ]:
def count_classes(split):
    counts = pd.Series(dtype=int)
    for f in sorted((DATA / split).glob("*.parquet")):
        col = pq.read_table(f, columns=["source_class"]).to_pandas()["source_class"]
        counts = counts.add(col.value_counts(), fill_value=0)
    return counts.sort_index().astype(int)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, split in zip(axes, ["train", "calibration", "validation"]):
    c = count_classes(split)
    ax.bar(range(len(c)), c.values)
    ax.set_xticks(range(len(c)))
    ax.set_xticklabels([CLASS_NAMES[i] for i in c.index], rotation=30)
    ax.set_title(f"{split} (n={c.sum()})")
plt.tight_layout()
plt.savefig(ROOT / "report/figures/fig1_class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

for split in ["train", "calibration", "validation"]:
    c = count_classes(split)
    real, ai = int(c.get(0, 0)), int(c.drop(0, errors="ignore").sum())
    print(f"{split:12s} real={real}  ai={ai}  ai_share={ai/(real+ai):.2%}")

## 3. Image size & format

Decode a random subset (full streaming would be slow). Look for size/format patterns that correlate with class — a leak we should neutralise during cleaning.

In [ ]:
def sample_rows(split, n=1500, seed_=0):
    """Random rows from a split. Returns list of (label, bytes)."""
    rng = np.random.default_rng(seed_)
    files = sorted((DATA / split).glob("*.parquet"))
    out = []
    per_file = max(1, n // len(files) + 1)
    for f in files:
        df = pq.read_table(f).to_pandas()
        if len(df) == 0:
            continue
        take = min(per_file, len(df))
        idx = rng.choice(len(df), size=take, replace=False)
        for i in idx:
            row = df.iloc[i]
            label = int(row["source_class"]) if "source_class" in df.columns else -1
            out.append((label, row["image"]))
    rng.shuffle(out)
    return out[:n]

recs = []
for label, buf in sample_rows("train", n=2000, seed_=1):
    try:
        with Image.open(io.BytesIO(buf)) as im:
            recs.append({"class": label, "w": im.width, "h": im.height,
                         "mode": im.mode, "format": im.format, "bytes": len(buf)})
    except Exception as e:
        recs.append({"class": label, "error": str(e)[:80]})

size_df = pd.DataFrame(recs)
print("modes:\n", size_df["mode"].value_counts(dropna=False), "\n")
print("formats:\n", size_df["format"].value_counts(dropna=False), "\n")
print("per-class size stats:\n", size_df.groupby("class")[["w", "h", "bytes"]]
      .agg(["median", "min", "max"]).round(0))

fig, ax = plt.subplots(figsize=(7, 6))
_, _, _, im = ax.hist2d(size_df["w"].dropna(), size_df["h"].dropna(), bins=40, cmin=1)
plt.colorbar(im, ax=ax, label="number of images in bin")
ax.set_xlabel("width (px)")
ax.set_ylabel("height (px)")
ax.set_title("image dimensions (train sample, n=2000)")
ax.text(
    0.02, 0.98,
    "Clusters:\n"
    "  (270, 270)  -> DALL-E3\n"
    "  (320, 320)  -> SD2.1 / SDXL / SD3 / Midjourney\n"
    "  scattered   -> real (variable, non-square)",
    transform=ax.transAxes, va="top", ha="left", fontsize=8, color="white",
    bbox=dict(boxstyle="round", facecolor="#111111", alpha=0.7),
)
plt.tight_layout()
plt.savefig(ROOT / "report/figures/fig2_image_dimensions.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Per-class descriptive stats

Mean/std RGB and file-size bytes per source class. If any one stat clearly separates real vs AI, the model could learn that shortcut — worth flagging in the report.

In [5]:
rows = []
for label, buf in sample_rows("train", n=600, seed_=2):
    try:
        a = np.asarray(Image.open(io.BytesIO(buf)).convert("RGB"), dtype=np.float32) / 255.0
        rows.append({
            "class": label,
            "mean_r": a[..., 0].mean(), "mean_g": a[..., 1].mean(), "mean_b": a[..., 2].mean(),
            "std": float(a.std()), "bytes": len(buf),
        })
    except Exception:
        pass

stat_df = pd.DataFrame(rows).groupby("class").mean().round(3)
stat_df.index = [CLASS_NAMES[i] for i in stat_df.index]
stat_df

,mean_r,mean_g,mean_b,std,bytes
real,0.463,0.450,0.421,0.247,50839.042
SD2.1,0.486,0.466,0.425,0.225,32449.865
SDXL,0.503,0.477,0.435,0.191,27039.577
SD3,0.560,0.547,0.490,0.270,29902.366
DALL-E3,0.482,0.452,0.410,0.251,18116.337
Midjourney,0.445,0.429,0.385,0.223,24340.343


## 5. Sample grid

One row per source class — visual sanity check.

In [ ]:
buckets = {c: [] for c in range(6)}
for label, buf in sample_rows("train", n=4000, seed_=3):
    if 0 <= label < 6 and len(buckets[label]) < 6:
        buckets[label].append(buf)
    if all(len(v) == 6 for v in buckets.values()):
        break

fig, axes = plt.subplots(6, 6, figsize=(12, 12))
for c in range(6):
    for j in range(6):
        ax = axes[c, j]
        ax.set_xticks([]); ax.set_yticks([])
        if j < len(buckets[c]):
            ax.imshow(Image.open(io.BytesIO(buckets[c][j])).convert("RGB"))
        if j == 0:
            ax.set_ylabel(CLASS_NAMES[c], fontsize=10)
plt.tight_layout()
plt.savefig(ROOT / "report/figures/fig3_sample_grid.png", dpi=150, bbox_inches="tight")
plt.show()

## Findings

### Dataset overview
- **Sizes**: train ~29,700 rows (7 files x 4,242); calibration ~1,924; validation ~1,124; predict 100.
- **Format**: every image is a JPEG in RGB mode, no format or mode variation whatsoever.
- All 6 source classes (0-5) confirmed present in the training data.

### Class balance
Consistent ~83% AI / 17% real across train, calibration, and validation. The ratio is stable across splits, so no split-specific reweighting is needed at this stage.

### Image dimensions - near-perfect leak
| Class | Observed size | Square? |
|---|---|---|
| real (0) | variable, median 640x480, min 320x201 | No |
| SD 2.1 (1) | exactly 320x320 | Yes |
| SDXL (2) | exactly 320x320 | Yes |
| SD 3 (3) | exactly 320x320 | Yes |
| DALL-E 3 (4) | exactly 270x270 | Yes |
| Midjourney (5) | exactly 320x320 | Yes |

Aspect ratio alone (`width == height`) is a near-perfect binary classifier. Within AI classes, the exact pixel size further separates DALL-E 3 from the rest. Any model trained on images that retain original dimensions would learn this metadata shortcut rather than image content.

### File size - secondary leak (dataset artifact, not a real feature)
Real images median ~50 KB. AI classes are substantially smaller: SD2.1 ~32 KB, SD3 ~30 KB, SDXL ~28 KB, Midjourney ~24 KB, DALL-E3 ~18 KB. This reflects different JPEG quality settings used by each generator, not image content. After converting to 224x224 float32 tensors the original file size is gone. Even if saved separately, it would not generalise to the hidden holdout if images were re-encoded at different quality settings. Not used as a feature.

### RGB descriptive statistics
- SD3 is notably warmer and brighter across all channels (mean_r=0.560 vs 0.44-0.50 for all others).
- SDXL has the lowest pixel std (0.191), textures are flatter and more uniform than in real photos.
- Channel means alone do not cleanly separate real from AI in general; dimensions and file size are stronger signals, but both are dataset artifacts rather than content features.

### Visual cues (from sample grid)

**Real**: random snapshots from everyday life, no special composition or camera quality. Varied subjects, natural imperfections, non-square framing. Unprofessional in the best sense, nothing feels intentional or staged.

**DALL-E 3**: most obviously AI to the human eye. Images look plastic, animated, or illustrated. Lighting is unnatural and scenes often have an uncanny "rendered" quality. Easiest class to spot at a glance.

**SD 2.1**: more realistic lighting than DALL-E 3, less of the animated look. The giveaway is structural/logical errors: wrong number of fingers, malformed faces, broken geometry in complex objects.

**SD 3**: similar to SD 2.1 in character, more realistic lighting, but still frequently obvious due to logical errors in complex structures and figures.

**SDXL**: fewer logical errors than the earlier SD models, but images look heavily filtered. Oversaturated, high contrast, artificial-feeling lighting, like an Instagram filter was applied on top. Less "broken" but clearly not a real photo.

**Midjourney**: hardest to detect. The best images look like they were shot with a high-quality camera, well-composed, film-like. That cinematic "too perfect" quality is actually the main tell compared to real snapshots. Does have outliers though, some images are obviously animated or stylised and give it away instantly.

**General tells (hardest to easiest for a human):**
1. Logical errors, wrong limb count, entangled body parts, impossible geometry -> most reliable giveaway across all AI models
2. Animated/plastic look (DALL-E 3 mainly) -> obvious on first glance
3. "Too perfect" composition and lighting -> filmlike quality that real casual photos rarely have

---

## Cleaning Decisions

The dataset is already clean: no corrupt images, no format variation, no missing labels. The only meaningful cleaning step is leak neutralisation via the image transform. This is preprocessing motivated by the findings above, not a response to data quality issues.

**1. Resize shorter-edge -> center-crop to 224x224**
Image dimensions are a near-perfect class signal (AI = square; DALL-E 3 = 270 px vs others = 320 px; real = non-square variable). Mapping all images to a fixed canvas neutralises this. Aspect-preserving resize avoids introducing stretch artefacts that could encode the original ratio. 224x224 matches Appendix B CNN and the reference timing script.

**2. Convert to RGB**
All images are already RGB JPEG, so this is defensive: ensures consistent 3-channel input regardless of edge cases in unseen data. Image mode must not be a learnable feature.

**3. Drop unreadable rows**
Corrupt bytes cause training failures. Smoke test found 0 corrupt rows in 500 sampled, so the drop rate is expected to be negligible, but the guard is required for correctness.

**4. No deduplication**
No cross-split near-duplicates identified. Will revisit if validation curves suggest train/val leakage during modelling.

---

## Implementation

In [7]:
IMG_SIZE = 224  # Appendix B CNN input; reference timing also uses this

def clean_image(buf):
    """bytes -> (224, 224, 3) float32 in [0, 1], or None if unreadable.

    Resize-shorter-edge + center-crop preserves aspect; we don't want stretching
    to encode original dimensions.
    """
    try:
        im = Image.open(io.BytesIO(buf)).convert("RGB")
    except Exception:
        return None
    w, h = im.size
    s = IMG_SIZE / min(w, h)
    im = im.resize((int(round(w * s)), int(round(h * s))), Image.BILINEAR)
    nw, nh = im.size
    left = (nw - IMG_SIZE) // 2
    top = (nh - IMG_SIZE) // 2
    im = im.crop((left, top, left + IMG_SIZE, top + IMG_SIZE))
    return np.asarray(im, dtype=np.float32) / 255.0

buf = sample_rows("train", n=1, seed_=4)[0][1]
out = clean_image(buf)
print("shape:", out.shape, "dtype:", out.dtype, "range:", (float(out.min()), float(out.max())))

shape: (224, 224, 3) dtype: float32 range: (0.01568627543747425, 1.0)


## Smoke test

Verify that every row in a 500-row sample produces a 224×224×3 float32 in [0, 1], or is reported as dropped.

In [8]:
n_total, n_dropped = 0, 0
shape_ok, range_ok = True, True
for _, buf in sample_rows("train", n=500, seed_=5):
    n_total += 1
    out = clean_image(buf)
    if out is None:
        n_dropped += 1
        continue
    if out.shape != (IMG_SIZE, IMG_SIZE, 3) or out.dtype != np.float32:
        shape_ok = False
    if out.min() < 0.0 or out.max() > 1.0:
        range_ok = False

assert shape_ok, "shape/dtype check failed"
assert range_ok, "value range check failed"
print(f"smoke test: {n_total} rows, {n_dropped} dropped, all kept rows are 224x224x3 float32 in [0, 1]")

smoke test: 500 rows, 0 dropped, all kept rows are 224x224x3 float32 in [0, 1]


## 8. Port targets — hand-off to `solution/`

When this notebook converges, port:

- `_lib/io.py::read_parquet_split(split_dir)` ← streaming version of `sample_rows` that yields all rows (no random sampling, no eager `to_pandas`)
- `_lib/io.py::decode_image(buf)` ← `Image.open(BytesIO(buf)).convert("RGB")` returning PIL or array; expose `IMG_SIZE` constant here
- `_lib/data.py::LabeledImageDataset.__iter__` ← combine the two helpers + `binarize_label(source_class)`
- `clean.py::explore_class_distribution` ← cell 2
- `clean.py::explore_image_sizes` ← cell 3
- `clean.py::explore_descriptive_stats` ← cell 4
- `clean.py::clean_pipeline` ← `clean_image()`
- `clean.py::save_cleaned_dataset` ← write cleaned tensors/parquet under `artifacts/clean/` (deferred until prepare.py is needed)

Figures from cells 2, 3, 5 land in report §1.1.